In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/content/vgsales.csv")

df.head()

In [ ]:
# Dataset information
df.info()
# Statistical summary
df.describe()

In [ ]:
# Dataset dimensions
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Column names
print("Columns in the dataset:")
print(df.columns.tolist())

In [ ]:
# Target variable
target = "Global_Sales"

print("Target variable:", target)
print("Mean Global Sales:", df[target].mean())
print("Median Global Sales:", df[target].median())
print("Minimum Global Sales:", df[target].min())
print("Maximum Global Sales:", df[target].max())

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(df["Global_Sales"], bins=50, kde=True)

plt.title("Distribution of Global Video Game Sales")
plt.xlabel("Global Sales (Millions)")
plt.ylabel("Number of Games")

plt.show()

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)

print(missing_values)

In [ ]:
missing_percentage = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

print(missing_percentage)

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

In [ ]:
df["Year"] = df["Year"].fillna(df["Year"].median())

df["Publisher"] = df["Publisher"].fillna("Unknown")

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df["Year"].describe())

In [ ]:
print("Minimum year:", df["Year"].min())
print("Maximum year:", df["Year"].max())

In [ ]:
plt.figure(figsize=(12, 5))

sns.histplot(df["Year"], bins=30)

plt.title("Distribution of Video Game Release Years")
plt.xlabel("Release Year")
plt.ylabel("Number of Games")

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="Genre",
    y="Global_Sales"
)

plt.title("Global Sales by Game Genre")
plt.xlabel("Genre")
plt.ylabel("Global Sales (Millions)")
plt.xticks(rotation=45)

plt.show()

In [ ]:
genre_sales = (
    df.groupby("Genre")["Global_Sales"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 6))

sns.barplot(
    x=genre_sales.values,
    y=genre_sales.index
)

plt.title("Average Global Sales by Genre")
plt.xlabel("Average Global Sales (Millions)")
plt.ylabel("Genre")

plt.show()

In [ ]:
top_platforms = df["Platform"].value_counts().head(15)

plt.figure(figsize=(12, 6))

sns.barplot(
    x=top_platforms.values,
    y=top_platforms.index
)

plt.title("Top 15 Platforms by Number of Games")
plt.xlabel("Number of Games")
plt.ylabel("Platform")

plt.show()

In [ ]:
platform_sales = (
    df.groupby("Platform")["Global_Sales"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))

sns.barplot(
    x=platform_sales.values,
    y=platform_sales.index
)

plt.title("Top Platforms by Average Global Sales")
plt.xlabel("Average Global Sales (Millions)")
plt.ylabel("Platform")

plt.show()

In [ ]:
yearly_sales = (
    df.groupby("Year")["Global_Sales"]
    .sum()
)

plt.figure(figsize=(12, 6))

plt.plot(
    yearly_sales.index,
    yearly_sales.values
)

plt.title("Total Global Video Game Sales by Year")
plt.xlabel("Year")
plt.ylabel("Total Global Sales (Millions)")

plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

correlation = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
top_games = df.sort_values(
    "Global_Sales",
    ascending=False
).head(10)

top_games[["Name", "Platform", "Genre", "Year", "Global_Sales"]]

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=top_games,
    x="Global_Sales",
    y="Name"
)

plt.title("Top 10 Best-Selling Video Games")
plt.xlabel("Global Sales (Millions)")
plt.ylabel("Game")

plt.show()

## Model Preparation

For this prediction task, the target variable is **Global_Sales**.

We will use the following features:
- Platform
- Year
- Genre
- Publisher

The **Name** column is excluded from model training because it contains a very large number of unique game titles. One-hot encoding it creates thousands of unnecessary features and makes model training inefficient.


In [ ]:
# Select features and target
features = [
    "Platform",
    "Year",
    "Genre",
    "Publisher"
]

X = df[features].copy()
y = df["Global_Sales"]

# Handle missing values
X["Year"] = X["Year"].fillna(X["Year"].median())
X["Publisher"] = X["Publisher"].fillna("Unknown")

print("Features used:", features)
print("Missing values after handling:")
print(X.isnull().sum())


In [ ]:
# One-hot encode categorical features
X_encoded = pd.get_dummies(
    X,
    columns=["Platform", "Genre", "Publisher"],
    drop_first=True
)

print("Original feature shape:", X.shape)
print("Encoded feature shape:", X_encoded.shape)
X_encoded.head()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Number of training features:", X_train.shape[1])


## Model 1: Random Forest Regressor

Random Forest is a tree-based ensemble model that can capture nonlinear relationships and interactions between the encoded game features.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Regressor Performance")
print("------------------------------------")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    rf_predictions,
    alpha=0.5
)

plt.xlabel("Actual Global Sales")
plt.ylabel("Predicted Global Sales")
plt.title("Random Forest: Actual vs Predicted Global Sales")

min_value = min(y_test.min(), rf_predictions.min())
max_value = max(y_test.max(), rf_predictions.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.show()


## Model 2: Gradient Boosting Regressor

Gradient Boosting builds an ensemble of decision trees sequentially, where each new tree attempts to improve the errors made by the previous trees.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_predictions = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_predictions)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_predictions))
gb_r2 = r2_score(y_test, gb_predictions)

print("Gradient Boosting Regressor Performance")
print("----------------------------------------")
print("MAE :", gb_mae)
print("RMSE:", gb_rmse)
print("R²  :", gb_r2)


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    gb_predictions,
    alpha=0.5
)

plt.xlabel("Actual Global Sales")
plt.ylabel("Predicted Global Sales")
plt.title("Gradient Boosting: Actual vs Predicted Global Sales")

min_value = min(y_test.min(), gb_predictions.min())
max_value = max(y_test.max(), gb_predictions.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.show()


## Model Performance Comparison

The two models are compared using:
- **MAE** — average absolute prediction error.
- **RMSE** — penalizes larger prediction errors more strongly.
- **R²** — proportion of variance explained by the model.


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Random Forest Regressor",
        "Gradient Boosting Regressor"
    ],
    "MAE": [rf_mae, gb_mae],
    "RMSE": [rf_rmse, gb_rmse],
    "R2": [rf_r2, gb_r2]
})

results


In [ ]:
plt.figure(figsize=(10, 6))

results_plot = results.set_index("Model")[["MAE", "RMSE"]]
results_plot.plot(kind="bar", ax=plt.gca())

plt.title("Model Error Comparison")
plt.xlabel("Model")
plt.ylabel("Error")
plt.xticks(rotation=0)
plt.legend(title="Metric")
plt.tight_layout()

plt.show()


## Model Interpretation

Both models use the same cleaned and encoded feature set. Random Forest provides feature importance values based on the reduction in impurity across its trees. Gradient Boosting also provides feature importance values based on its fitted trees.


In [ ]:
# Random Forest feature importance
rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

rf_importance.head(15)


In [ ]:
# Gradient Boosting feature importance
gb_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gb_model.feature_importances_
}).sort_values("Importance", ascending=False)

gb_importance.head(15)
